# Step 05 - Bronze Layer

Mục tiêu:
- Đọc dữ liệu CSV từ MinIO landing zone
- Giữ dữ liệu gần với dữ liệu nguồn
- Thêm metadata ingestion
- Ghi dữ liệu thành Apache Iceberg tables trong namespace bronze

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [2]:
ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:"
    f"iceberg-spark-runtime-4.1_2.13:"
    f"{ICEBERG_VERSION},"
    f"org.apache.iceberg:"
    f"iceberg-aws-bundle:"
    f"{ICEBERG_VERSION},"
    f"org.apache.hadoop:"
    f"hadoop-aws:"
    f"{HADOOP_VERSION}"
)

spark = (
    SparkSession.builder
    .appName("EcommerceBronzeLayer")
    .master("local[*]")

    # Packages
    .config(
        "spark.jars.packages",
        PACKAGES
    )

    # Iceberg
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions."
        "IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )

    .config(
        "spark.sql.catalog.lakehouse.type",
        "rest"
    )

    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )

    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )

    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )

    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )

    # Hadoop S3A -> MinIO
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1"
    )

    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )

    .config(
        "spark.hadoop.fs.s3a.access.key",
        os.environ["AWS_ACCESS_KEY_ID"]
    )

    .config(
        "spark.hadoop.fs.s3a.secret.key",
        os.environ["AWS_SECRET_ACCESS_KEY"]
    )

    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .getOrCreate()
)

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f4cb8af8-639d-4738-bda7-ac40a3a9d2dd;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.11.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 288ms :: artifacts dl 12ms
	:: modules 

Spark version: 4.1.3


In [3]:
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "s3a://ecommerce-lakehouse/"
        "landing/orders/orders.csv"
    )
)

In [4]:
orders_bronze_df = (
    orders_df
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.input_file_name()
    )
)

In [5]:
(
    orders_bronze_df
    .writeTo("lakehouse.bronze.orders")
    .using("iceberg")
    .createOrReplace()
)

print("Đã tạo lakehouse.bronze.orders")

Đã tạo lakehouse.bronze.orders


In [6]:
orders_bronze_df.show(5,truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+---------------------------------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at              |_source_file                                       |
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+---------------------------------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11

In [7]:
BUCKET = "ecommerce-lakehouse"

DATASETS = {
    "customers":
        f"s3a://{BUCKET}/landing/customers/customers.csv",

    "products":
        f"s3a://{BUCKET}/landing/products/products.csv",

    "orders":
        f"s3a://{BUCKET}/landing/orders/orders.csv",

    "order_items":
        f"s3a://{BUCKET}/landing/order_items/order_items.csv",

    "reviews":
        f"s3a://{BUCKET}/landing/reviews/reviews.csv",

    "payments":
        f"s3a://{BUCKET}/landing/payments/payments.csv",
}

In [8]:
from pyspark.sql import functions as F

def load_to_bronze(dataset_name, source_path):
    print(f"\n=== Processing: {dataset_name} ===")

    # 1. Tạo CSV reader cơ bản
    reader = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
    )

    # 2. Reviews có text phức tạp:
    # dấu phẩy, dấu ngoặc kép, xuống dòng
    if dataset_name == "reviews":
        reader = (
            reader
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", "true")
        )

    # 3. Đọc CSV
    df = reader.csv(source_path)

    # 4. Thêm metadata Bronze
    bronze_df = (
        df
        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.input_file_name()
        )
    )

    # 5. Tên Iceberg table
    target_table = f"lakehouse.bronze.{dataset_name}"

    # 6. Ghi vào Bronze
    (
        bronze_df
        .writeTo(target_table)
        .using("iceberg")
        .createOrReplace()
    )

    # 7. Kiểm tra số dòng
    row_count = bronze_df.count()

    print(f"✅ Created: {target_table}")
    print(f"Rows: {row_count:,}")

In [9]:
for dataset_name, source_path in DATASETS.items():
    load_to_bronze(
        dataset_name,
        source_path
    )


=== Processing: customers ===


✅ Created: lakehouse.bronze.customers
Rows: 99,441

=== Processing: products ===
✅ Created: lakehouse.bronze.products
Rows: 32,951

=== Processing: orders ===


✅ Created: lakehouse.bronze.orders
Rows: 99,441

=== Processing: order_items ===
✅ Created: lakehouse.bronze.order_items
Rows: 112,650

=== Processing: reviews ===
✅ Created: lakehouse.bronze.reviews
Rows: 99,224

=== Processing: payments ===
✅ Created: lakehouse.bronze.payments
Rows: 103,886


In [10]:
spark.sql("""
show tables in lakehouse.bronze
""").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|   bronze|  customers|      false|
|   bronze|order_items|      false|
|   bronze|     orders|      false|
|   bronze|   payments|      false|
|   bronze|   products|      false|
|   bronze|    reviews|      false|
+---------+-----------+-----------+

